In [24]:
import pandas as pd

df = pd.read_excel('/Users/gabriellabollicimoreno/Desktop/DEPRESSION VLOGS/Spanish_Data_Gabriella.xlsx', sheet_name='Sheet1')
df = df.dropna(how='all')
df = df[["id", "Transcript"]]
df.head(10)

,id,Transcript
0,0.0,[Música] [Música] k [Música] bueno en primer l...
1,1.0,Hola a todos y bienvenidos un domingo más a mi...
2,2.0,ah ah estoy superado tanto este momento hola a...
3,3.0,hola a todos ya todas y bienvenidos a mi canal...
4,4.0,ahora a todos ya todas y bienvenidos a mi cana...
5,5.0,hola a todos ya todas y bienvenidos de nuevo o...
6,6.0,hola a todos ya todas y bienvenidos de nuevo o...
7,7.0,hola a todos ya todas y bienvenidos a mi canal...
8,8.0,[Música] [Aplausos] allah [Música] ah Hola chi...
9,9.0,Hola Yo soy clara Cuevas Bienvenido a mi canal...


In [3]:
pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.1.0/es_core_news_sm-3.1.0.tar.gz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 23.7 MB/s eta 0:00:00 0:00:01
  Preparing metadata (setup.py) ... done
  Created wheel for es_core_news_sm: filename=es_core_news_sm-3.1.0-py3-none-any.whl size=13675550 sha256=40195b09f3f26ed48976a58e266a28ef576ddbe885d37b1fe94b943c0ca5eec2
  Stored in directory: /Users/gabriellabollicimoreno/Library/Caches/pip/wheels/97/7f/78/e75df40afebcd9a58c409b6ae3942e26ef2b4a8ba4c2d8ce1a
Successfully built es_core_news_sm
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from langdetect import detect

detected_language = detect(' '.join(df['Transcript'].astype(str)))

if detected_language == 'en':
    new_column_name = 'Transcript_english'
elif detected_language == 'es':
    new_column_name = 'Transcript_spanish'
elif detected_language == 'nl':
    new_column_name = 'Transcript_dutch'
elif detected_language == 'tr':
    new_column_name = 'Transcript_turkish'
else:
    new_column_name = f'Transcript_{detected_language}'

df.rename(columns={'Transcript': new_column_name}, inplace=True)
df = df.head(130)

display(df)

,id,Transcript_spanish
0,0.0,[Música] [Música] k [Música] bueno en primer l...
1,1.0,Hola a todos y bienvenidos un domingo más a mi...
2,2.0,ah ah estoy superado tanto este momento hola a...
3,3.0,hola a todos ya todas y bienvenidos a mi canal...
4,4.0,ahora a todos ya todas y bienvenidos a mi cana...
...,...,...
125,125.0,Hola cachos de [ __ ] Antes que nada decir que...
126,126.0,hola cachos de [ __ ] que nada que he vuelto p...
127,127.0,hola cachos de [ __ ] mira qué fantasía gregg ...
128,128.0,menos mal que me he dado cuenta que estaba el ...


This code processes text data from a DataFrame column that contains language-specific transcripts, identified by the column name format Transcript_language (e.g., Transcript_english). It starts by extracting the language from the column name and loads the corresponding spaCy model for that language (e.g., English, Spanish, French). The code then cleans the text by removing any content inside brackets or parentheses. Using the spaCy model, it tokenizes the text, performs Part-of-Speech (POS) tagging, lemmatizes the tokens if available, and extracts named entities. Finally, it updates the DataFrame with new columns containing the cleaned text, tokens, POS tags, lemmas, and recognized entities.

In [3]:
import re
import os
import spacy
import pandas as pd

transcript_column = [col for col in df.columns if col.startswith('Transcript_')][0]
detected_language = transcript_column.split('_')[1]

if detected_language == 'english':
    nlp = spacy.load("en_core_web_sm")
elif detected_language == 'spanish':
    nlp = spacy.load("es_core_news_sm")
elif detected_language == 'dutch':
    nlp = spacy.load("nl_core_news_sm")
else:
    raise ValueError(f"spaCy model is not available for the detected language: {detected_language}")

# Function to clean and remove text between brackets 
def remove_text_in_brackets(text):
    if isinstance(text, str):
        return re.sub(r'[<\[\(\{\}\)\]>]', '', text)
    else:
        return ''

df['clean_transcript'] = df[transcript_column].apply(remove_text_in_brackets)

# Tokenization and text analysis
def tokenize_and_pos_tag(text):
    doc = nlp(text) 
    tokens = [token.text for token in doc]
    pos_tags = [(token.text, token.pos_) for token in doc]
    
    if 'lemmatizer' in nlp.pipe_names:
        lemma = [(token.text, token.lemma_) for token in doc]
    else:
        lemma = [(token.text, None) for token in doc]
    
    entity = [(ent.text, ent.start_char, ent.end_char, ent.label_) for ent in doc.ents]
    
    return tokens, pos_tags, lemma, entity

df[['tokens', 'pos_tag', 'lemma', 'entity']] = df['clean_transcript'].apply(lambda text: pd.Series(tokenize_and_pos_tag(text)))

def split_into_phrases(text):
    if isinstance(text, str):
        doc = nlp(text)
        return [sent.text.strip() for sent in doc.sents]
    return []

df['phrases'] = df['clean_transcript'].apply(split_into_phrases)
display(df)

/opt/homebrew/lib/python3.11/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'es_core_news_sm' (3.1.0) was trained with spaCy v3.1.0 and may not be 100% compatible with the current version (3.8.2). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


,id,Transcript_spanish,clean_transcript,tokens,pos_tag,lemma,entity,phrases
0,0.0,[Música] [Música] k [Música] bueno en primer l...,Música Música k Música bueno en primer lugar H...,"[Música, Música, k, Música, bueno, en, primer,...","[(Música, PROPN), (Música, PROPN), (k, ADP), (...","[(Música, Música), (Música, Música), (k, k), (...","[(Música Música, 0, 13, LOC), (Música, 16, 22,...",[Música Música k Música bueno en primer lugar ...
1,1.0,Hola a todos y bienvenidos un domingo más a mi...,Hola a todos y bienvenidos un domingo más a mi...,"[Hola, a, todos, y, bienvenidos, un, domingo, ...","[(Hola, PROPN), (a, ADP), (todos, PRON), (y, C...","[(Hola, Hola), (a, a), (todos, todo), (y, y), ...","[(Hola, 0, 4, PER), (mí, 143, 145, PER), (Dios...",[Hola a todos y bienvenidos un domingo más a m...
2,2.0,ah ah estoy superado tanto este momento hola a...,ah ah estoy superado tanto este momento hola a...,"[ah, ah, estoy, superado, tanto, este, momento...","[(ah, ADV), (ah, ADV), (estoy, AUX), (superado...","[(ah, ah), (ah, ah), (estoy, estar), (superado...","[(youtube, 99, 106, MISC), (súper, 204, 209, P...",[ah ah estoy superado tanto este momento hola ...
3,3.0,hola a todos ya todas y bienvenidos a mi canal...,hola a todos ya todas y bienvenidos a mi canal...,"[hola, a, todos, ya, todas, y, bienvenidos, a,...","[(hola, PROPN), (a, ADP), (todos, PRON), (ya, ...","[(hola, hola), (a, a), (todos, todo), (ya, ya)...","[(más, 58, 61, ORG), (mí, 2681, 2683, PER), (m...",[hola a todos ya todas y bienvenidos a mi cana...
4,4.0,ahora a todos ya todas y bienvenidos a mi cana...,ahora a todos ya todas y bienvenidos a mi cana...,"[ahora, a, todos, ya, todas, y, bienvenidos, a...","[(ahora, ADV), (a, ADP), (todos, PRON), (ya, A...","[(ahora, ahora), (a, a), (todos, todo), (ya, y...","[(grecia, 75, 81, LOC), (mí, 482, 484, PER), (...",[ahora a todos ya todas y bienvenidos a mi can...
...,...,...,...,...,...,...,...,...
125,125.0,Hola cachos de [ __ ] Antes que nada decir que...,Hola cachos de __ Antes que nada decir que t...,"[Hola, cachos, de, , _, _, , Antes, que, nad...","[(Hola, PROPN), (cachos, NOUN), (de, ADP), ( ,...","[(Hola, Hola), (cachos, cacho), (de, de), ( , ...","[(Hola, 0, 4, PER), (Club, 83, 87, ORG), (Bárb...",[Hola cachos de __ Antes que nada decir que ...
126,126.0,hola cachos de [ __ ] que nada que he vuelto p...,hola cachos de __ que nada que he vuelto per...,"[hola, cachos, de, , _, _, , que, nada, que,...","[(hola, PROPN), (cachos, NOUN), (de, ADP), ( ,...","[(hola, hola), (cachos, cacho), (de, de), ( , ...","[(sofá, 154, 158, LOC), (brisita, 166, 173, OR...",[hola cachos de __ que nada que he vuelto pe...
127,127.0,hola cachos de [ __ ] mira qué fantasía gregg ...,hola cachos de __ mira qué fantasía gregg ar...,"[hola, cachos, de, , _, _, , mira, qué, fant...","[(hola, PROPN), (cachos, NOUN), (de, ADP), ( ,...","[(hola, hola), (cachos, cacho), (de, de), ( , ...","[(leonardo dicaprio, 150, 167, PER), (romeo y ...",[hola cachos de __ mira qué fantasía gregg a...
128,128.0,menos mal que me he dado cuenta que estaba el ...,menos mal que me he dado cuenta que estaba el ...,"[menos, mal, que, me, he, dado, cuenta, que, e...","[(menos, ADV), (mal, ADV), (que, SCONJ), (me, ...","[(menos, menos), (mal, mal), (que, que), (me, ...","[(scarlett johansson hola, 84, 107, PER), (mí ...",[menos mal que me he dado cuenta que estaba el...


In [4]:
mental_health_words_sp = """triste, feliz, desesperado, esperanzado, vacío, satisfecho, insatisfecho, alto, bajo,
entumecido, sensible, melancólico, alegre, derrotado, exitoso, triste, agradable, azul, optimista, 
sombrío, brillante, desanimado, inspirado, sin alegría, alegre, insensible, sensible, indiferente, preocupado, 
desmotivado, motivado, desvinculado, comprometido, aburrido, interesado, desvinculado, implicado, insatisfecho, 
realizado, sin vida, animado, desconectado, conectado, en blanco, expresivo, sin emoción, emocional, culpable, 
inocente, sin valor, valioso, avergonzado, orgulloso, fracasado, exitoso, inadecuado, capaz, autoculpado, 
autoaceptación, arrepentido, contento, inútil, útil, defectuoso, perfecto, vergüenza, asertivo, indigno, 
digno, indigno, merecedor, cansado, energizado, agotado, refrescado, drenado, recargado, fatigado, descansado, 
perezoso, activo, enérgico, vibrante, somnoliento, alerta, agotado, revitalizado, quemado, renovado, ansioso, 
tranquilo, nervioso, seguro de sí mismo, inquieto, nervioso, compuesto, temeroso, valiente, tenso, tranquilo, 
preocupado, seguro, agitado, presa del pánico, frío, inquieto, angustiado, preocupado, a gusto, distraído, atento, 
olvidado, atento, indeciso, decisivo, desenfocado, centrado, claro, nublado, lúcido, disperso, organizado, confuso,
seguro, perdido, aterrizado, nebuloso, agudo, ausente, despierto, dolor de cabeza, alivio, náuseas, dolor muscular, 
flexibilidad, dolor, facilidad, dolor, débil, fuerte, mareado, firme, dolorido, tenso, flojo, dolor corporal, bienestar,
insomnio, insomne, dormido, dormido de más, interrumpido, ininterrumpido, pesadilla, roto, dormido, privado de sueño, 
bien descansado, inapetente, apetito, antojos, satisfacción, atracón, nutrido, bajo peso, sano, equilibrado, 
malnutrido, aislado, solo, junto, retirado, rechazado, aceptado, no querido, amado, incomprendido, entendido,
descuidado, cuidado, evitado, abrazado, sin sentido, precioso, significativo, desolado, sin sentido, significativo,
fútil, fructífero, completo, logro, confianza, pesimista, animado, condenado, resistente, sin propósito, impulsado,
inseguro, seguro, incompetente, competente"""

mental_health_words_nl = [word.strip().lower() for word in mental_health_words_sp.split(',')]

Low level sentiment analysis. https://ojs.aaai.org/index.php/ICWSM/article/view/14550
Timestamp (track the development of sentiment for every vlogger)
Check for approximate number of words
How likely are people from different cultures to express emotions? 

NER: mental conditions to see if they correlate with depression. LLMs?? LlaMA3? 
List of things that have not been done and should be done. 

In [5]:
import spacy
from spacy.matcher import PhraseMatcher
from spacy.tokens import Span

# Inicializar Spacy y el PhraseMatcher
nlp = spacy.load("es_core_news_sm")
matcher = PhraseMatcher(nlp.vocab)
patterns = [nlp.make_doc(word) for word in mental_health_words_nl]
matcher.add("MENTAL_HEALTH", patterns)

# Función para detectar las entidades relacionadas con salud mental
def detect_mental_health_entities(doc):
    matches = matcher(doc)
    spans = [Span(doc, start, end, label="MENTAL_HEALTH") for match_id, start, end in matches]
    spans = spacy.util.filter_spans(spans)
    existing_ents = list(doc.ents)
    non_conflicting_spans = [span for span in spans if not any(ent.start < span.end and span.start < ent.end for ent in existing_ents)]
    doc.ents = existing_ents + non_conflicting_spans

    mental_health_entities = [(ent.text, ent.label_) for ent in doc.ents if ent.label_ == "MENTAL_HEALTH"]
    return mental_health_entities

# Aplicar la detección de entidades al DataFrame
df['mental_health_entities'] = df['clean_transcript'].apply(
    lambda text: detect_mental_health_entities(nlp(text))
)

# Mostrar el DataFrame con la nueva columna
output_file = '/Users/gabriellabollicimoreno/Desktop/DEPRESSION VLOGS/Datasets/Spanish/Spanish_Dataset_Processed.csv'
os.makedirs(os.path.dirname(output_file), exist_ok=True)  # Ensure the directory exists
df.to_csv(output_file, index=False)

print(f"File has been saved as: {output_file}")

/opt/homebrew/lib/python3.11/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'es_core_news_sm' (3.1.0) was trained with spaCy v3.1.0 and may not be 100% compatible with the current version (3.8.2). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


File has been saved as: /Users/gabriellabollicimoreno/Desktop/DEPRESSION VLOGS/Datasets/Spanish/Spanish_Dataset_Processed.csv


In [ ]:
!pip install unidecode

In [8]:
#Pronouns
FPS = ["yo", "me", "mí", "conmigo"]
FPP = ["nosotros", "nosotras", "nos"]
SPS = ["tú", "te", "ti", "contigo", "usted", "le", "lo", "la"]
SPP = ["ustedes", "les", "los", "las", "vosotros", "vosotras", "os"]
TPS = ["él", "lo", "le", "se", "consigo", "ella", "la", "le", "se", "consigo", "ello"]
TPP = ["ellos", "los", "les", "se", "ellas", "las", "les", "se"]

# Verb terminations for regular verbs
FPS_verbs = ["-o", "-é", "-aré", "-iré", "-aría", "-iera", "-iese"]
FPP_verbs = ["-amos", "-emos", "-imos", "-ábamos", "-éramos", "-íamos"]
SPS_verbs = ["-as", "-es", "-aste", "-iste", "-arás", "-erás", "-irás"]
SPP_verbs = ["-áis", "-éis", "-ís", "-aron", "-ieron", "-arán", "-erán", "-irán"]
TPS_verbs = ["-a", "-e", "-ó", "-ará", "-erá", "-irá"]
TPP_verbs = ["-an", "-en", "-aron", "-ieron", "-arán", "-erán", "-irán"]

# Most common irregular verbs
irregular_verbs_fps = ["soy", "voy", "tengo", "digo", "hago", "puedo", "quiero", "sé", "veo"]
irregular_verbs_fpp = ["somos", "vamos", "tenemos", "decimos", "hacemos", "podemos", "queremos", "sabemos", "vemos"]
irregular_verbs_sps = ["eres", "vas", "tienes", "dices", "haces", "puedes", "quieres", "sabes", "ves"]
irregular_verbs_spp = ["sois", "vais", "tenéis", "decís", "hacéis", "podéis", "queréis", "sabéis", "veis"]
irregular_verbs_tps = ["es", "va", "tiene", "dice", "hace", "puede", "quiere", "sabe", "ve"]
irregular_verbs_tpp = ["son", "van", "tienen", "dicen", "hacen", "pueden", "quieren", "saben", "ven"]

# Dictionary for pronouns and verbs
pronouns_dict = {
    "FPS": FPS,
    "FPP": FPP,
    "SPS": SPS,
    "SPP": SPP,
    "TPS": TPS,
    "TPP": TPP,
}

verbs_dict = {
    "FPS_verbs": FPS_verbs + irregular_verbs_fps,
    "FPP_verbs": FPP_verbs + irregular_verbs_fpp,
    "SPS_verbs": SPS_verbs + irregular_verbs_sps,
    "SPP_verbs": SPP_verbs + irregular_verbs_spp,
    "TPS_verbs": TPS_verbs + irregular_verbs_tps,
    "TPP_verbs": TPP_verbs + irregular_verbs_tpp,
}

results = []

for index, row in df.iterrows():
    row_data = {"clean_transcript": row["clean_transcript"]}
    
    # Count pronouns
    for category, pronouns in pronouns_dict.items():
        counts = {p: 0 for p in pronouns}
        pronoun_list = []
        for token, tag in zip(row['tokens'], row['pos_tag']):
            if "PRON" in tag and token.lower() in pronouns:
                counts[token.lower()] += 1
                pronoun_list.append(token)

        row_data[f"{category.lower()}_counts"] = [(k, v) for k, v in counts.items()]
        row_data[category] = sum(counts.values())
    
    # Count verbs
    for category, verbs in verbs_dict.items():
        verb_counts = []
        for token, tag in zip(row['tokens'], row['pos_tag']):
            if "VERB" in tag and (token.endswith(tuple(verbs)) or token.lower() in verbs):
                verb_counts.append(token)
        row_data[category] = verb_counts
    
    results.append(row_data)

df_pronouns = pd.DataFrame(results)

output_file_2 = '/Users/gabriellabollicimoreno/Desktop/DEPRESSION VLOGS/Datasets/Spanish/Spanish_Pronouns_Dataset.csv'
os.makedirs(os.path.dirname(output_file_2), exist_ok=True)  # Asegura que el directorio exista
df_pronouns.to_csv(output_file_2, index=False)

print(f"File has been saved as: {output_file_2}")
display(df_pronouns)

File has been saved as: /Users/gabriellabollicimoreno/Desktop/DEPRESSION VLOGS/Datasets/Spanish/Spanish_Pronouns_Dataset.csv


,clean_transcript,fps_counts,FPS,fpp_counts,FPP,sps_counts,SPS,spp_counts,SPP,tps_counts,TPS,tpp_counts,TPP,FPS_verbs,FPP_verbs,SPS_verbs,SPP_verbs,TPS_verbs,TPP_verbs
0,Música Música k Música bueno en primer lugar H...,"[(yo, 82), (me, 144), (mí, 22), (conmigo, 2)]",250,"[(nosotros, 0), (nosotras, 0), (nos, 1)]",1,"[(tú, 0), (te, 5), (ti, 1), (contigo, 0), (ust...",86,"[(ustedes, 8), (les, 21), (los, 0), (las, 0), ...",29,"[(él, 2), (lo, 71), (le, 9), (se, 31), (consig...",113,"[(ellos, 3), (los, 0), (les, 21), (se, 31), (e...",55,"[digo, tengo, puedo, Quiero, sé, sé, tengo, te...","[tenemos, vamos, vemos]",[llevas],[],"[hace, tiene, mueve, mueve, ayudarles, ayudarl...","[saben, saben, saben, hacen, saben, van, saben..."
1,Hola a todos y bienvenidos un domingo más a mi...,"[(yo, 69), (me, 188), (mí, 20), (conmigo, 0)]",277,"[(nosotros, 1), (nosotras, 0), (nos, 4)]",5,"[(tú, 0), (te, 30), (ti, 1), (contigo, 1), (us...",132,"[(ustedes, 1), (les, 39), (los, 6), (las, 3), ...",49,"[(él, 7), (lo, 89), (le, 10), (se, 40), (consi...",151,"[(ellos, 0), (los, 6), (les, 39), (se, 40), (e...",88,"[quiero, tengo, sé, digo, tengo, quiero, quier...","[vamos, vamos, vamos, vamos, vamos, vamos]","[tienes, tienes, tienes, Tienes, tienes, tiene...",[],"[darles, contarles, hace, hace, tienes, tienes...","[hacen, van, saben, saben, van, tienen, tienen..."
2,ah ah estoy superado tanto este momento hola a...,"[(yo, 56), (me, 142), (mí, 31), (conmigo, 2)]",231,"[(nosotros, 0), (nosotras, 0), (nos, 8)]",8,"[(tú, 0), (te, 3), (ti, 0), (contigo, 0), (ust...",70,"[(ustedes, 5), (les, 15), (los, 0), (las, 1), ...",21,"[(él, 2), (lo, 59), (le, 3), (se, 27), (consig...",98,"[(ellos, 0), (los, 0), (les, 15), (se, 27), (e...",43,"[tengo, quiero, quiero, tengo, quiero, quiero,...","[queremos, hacemos, vemos, vemos]","[vives, sabes]",[sabéis],"[hacerles, explicarles, hablarles, expones, re...","[hacen, saben, tienen, saben, tienen, hacen, v..."
3,hola a todos ya todas y bienvenidos a mi canal...,"[(yo, 8), (me, 30), (mí, 3), (conmigo, 0)]",41,"[(nosotros, 0), (nosotras, 0), (nos, 0)]",0,"[(tú, 2), (te, 0), (ti, 0), (contigo, 0), (ust...",38,"[(ustedes, 3), (les, 2), (los, 0), (las, 0), (...",5,"[(él, 0), (lo, 33), (le, 2), (se, 9), (consigo...",47,"[(ellos, 1), (los, 0), (les, 2), (se, 9), (ell...",12,"[tengo, tengo, sé, digo, tengo, quiero, quiero]",[],[],[],"[explicarles, hace, hace, tiene, lleva, lleva,...","[saben, saben, saben]"
4,ahora a todos ya todas y bienvenidos a mi cana...,"[(yo, 45), (me, 77), (mí, 26), (conmigo, 0)]",148,"[(nosotros, 2), (nosotras, 0), (nos, 10)]",12,"[(tú, 11), (te, 34), (ti, 3), (contigo, 0), (u...",118,"[(ustedes, 8), (les, 31), (los, 7), (las, 2), ...",49,"[(él, 0), (lo, 64), (le, 4), (se, 29), (consig...",100,"[(ellos, 2), (los, 7), (les, 31), (se, 29), (e...",71,"[veo, quiero, digo, quiero, hago, digo, digo, ...","[tenemos, tenemos, tenemos, hacemos, tenemos, ...","[haces, tienes, tienes, tienes, tienes, tienes...",[],"[hablarles, tiene, tiene, tiene, quiere, lleve...","[tienen, hacen, tienen, tienen, dicen]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,Hola cachos de __ Antes que nada decir que t...,"[(yo, 73), (me, 159), (mí, 18), (conmigo, 2)]",252,"[(nosotros, 1), (nosotras, 0), (nos, 8)]",9,"[(tú, 16), (te, 55), (ti, 0), (contigo, 0), (u...",170,"[(ustedes, 0), (les, 5), (los, 0), (las, 0), (...",12,"[(él, 3), (lo, 72), (le, 14), (se, 32), (consi...",136,"[(ellos, 3), (los, 0), (les, 5), (se, 32), (el...",40,"[tengo, digo, tengo, tengo, quiero, tengo, sé,...","[tenemos, volvemos, vamos]","[quieres, ves, sabes, ves, quieres, vas, quier...","[queréis, vais, hacéis, veis, tenéis, sabéis]","[ve, quiere, dice, dice, quieres, ves, tiene, ...","[tienen, tienen, llevan, hacen, llevan, van, h..."
126,hola cachos de __ que nada que he vuelto per...,"[(yo, 37), (me, 61), (mí, 2), (conmigo, 0)]",100,"[(nosotros, 0), (nosotras, 0), (nos, 0)]",0,"[(tú, 2), (te, 4), (ti, 0), (contigo, 0), (ust.